1、image preprocessing

In [ ]:
import tifffile as tiff
import cv2
import numpy as np
import os
import pandas as pd
def crop_visium_sample(sample_dir, out_dir, patch_size=224):
    
    img_path = None
    for f in os.listdir(sample_dir):
        if f.endswith(".tif") or f.endswith(".tiff"):
            img_path = os.path.join(sample_dir, f)
            break
    if img_path is None:
        print("No tif found", sample_dir)
        return
    img = tiff.imread(img_path)  
    if img.ndim == 2: 
        img = np.stack([img]*3, axis=-1)
    h, w = img.shape[:2]
    
    pos_path = os.path.join(sample_dir, "spatial", "tissue_positions_list.csv")
    if not os.path.exists(pos_path):
        pos_path = os.path.join(sample_dir, "spatial", "tissue_positions.csv")
    pos = pd.read_csv(pos_path, header=None)
    pos = pos[pos[1] == 1]  
    half = patch_size // 2
    os.makedirs(out_dir, exist_ok=True)
    for _, row in pos.iterrows():
        barcode = row[0]
        y = int(row[4])  # row
        x = int(row[5])  # col
        if x-half < 0 or y-half < 0 or x+half > w or y+half > h:
            continue
        patch = img[y-half:y+half, x-half:x+half]
        if patch.shape[0] != patch_size or patch.shape[1] != patch_size:
            continue
        
        cv2.imwrite(os.path.join(out_dir, f"{barcode}.jpg"), patch)
    print("Finished:", sample_dir)


def crop_all(root_dir):
    for sample in ["1142243F", "CID4290", "CID4465", "CID44971", "CID4535", "1160920F"]:
        sample_dir = os.path.join(root_dir, sample)
        if not os.path.isdir(sample_dir):
            continue
        out_dir = os.path.join(root_dir, "preprocessed_data", "ST-patches", sample)
        crop_visium_sample(sample_dir, out_dir)

# 运行
crop_all(".")

Finished: ./1142243F
Finished: ./CID4290
Finished: ./CID4465
Finished: ./CID44971
Finished: ./CID4535
Finished: ./1160920F


In [ ]:
import cv2


img1 = cv2.imread('/d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_data/ST-patches/1142243F/AAACAAGTATCTCCCA-1.jpg')


height1, width1, channels1 = img1.shape

print(f"图片尺寸: {width1} x {height1} 像素")
print(f"通道数: {channels1}")

import cv2

# 读取图片
img2 = cv2.imread('/d/zhoujl/my_model/dataset/her2st/preprocessed_data/patches_nmzd/A1/3x14.jpg')

# 获取图片尺寸（高度, 宽度, 通道数）
height2, width2, channels2 = img2.shape

print(f"图片尺寸: {width2} x {height2} 像素")
print(f"通道数: {channels2}")

图片尺寸: 224 x 224 像素
通道数: 3
图片尺寸: 224 x 224 像素
通道数: 3


In [10]:
# Reinhard color normalization for HE-stained histological images using histomicsTK tools.
import os
import PIL
import skimage.io
import skimage.color
import histomicstk as htk

def nmzd_reinhard_rescale(input_image_file, nmzd_path, barcode):
    """
    Reinhard图像颜色标准化
    使用 'ref_HE.png' 作为参考
    """
    rescale_size = 200
    im_input = skimage.io.imread(input_image_file)[:, :, :3]
    # 导入参考图像
    ref_image_file = 'ref_HE.png' 
    im_reference = skimage.io.imread(ref_image_file)[:, :, :3]
   
    mean_ref, std_ref = htk.preprocessing.color_conversion.lab_mean_std(im_reference)
    
    im_nmzd = htk.preprocessing.color_normalization.reinhard(im_input, mean_ref, std_ref)
    pil_img = PIL.Image.fromarray(im_nmzd)
   
    pil_img.save(os.path.join(nmzd_path, barcode+".jpg"))


directory_path = '/d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_data/ST-patches'
# 获取目录下的所有文件和文件夹
all_items = os.listdir(directory_path)
# 过滤出文件夹
tissue_list = [item for item in all_items if os.path.isdir(os.path.join(directory_path, item))]
print(len(tissue_list))

for tissue_name in tissue_list:
    source_path = os.path.join(directory_path,tissue_name)
    save_root_path = "/d/zhoujl/my_model/dataset/Alex_NatGen/patches_nmzd"
    nmzd_path = os.path.join(save_root_path, tissue_name)
    if not os.path.exists(nmzd_path):
        os.makedirs(nmzd_path)

    for filename in os.listdir(source_path):
        if filename.endswith('jpg'):
            try:
                barcode = filename[:-4]
                input_img_file = os.path.join(source_path, filename)
                nmzd_reinhard_rescale(input_img_file, nmzd_path, barcode)

            except:
                print("Error occured in %s" % os.path.join(source_path, filename))
    
    print("End of normalization & rescaling of %s" % tissue_name)

6
End of normalization & rescaling of CID4535
End of normalization & rescaling of 1142243F
End of normalization & rescaling of 1160920F
End of normalization & rescaling of CID4290
End of normalization & rescaling of CID4465
End of normalization & rescaling of CID44971


In [ ]:
cd Hover-net/hover_net-master/hover_net-master/

python run_infer.py \
--gpu='0' \
--nr_types=6 \
--type_info_path=type_info.json \
--batch_size=64 \
--model_mode=fast \
--model_path=pretrained/hovernet_fast_pannuke_type_tf2pytorch.tar \
--nr_inference_workers=8 \
--nr_post_proc_workers=16 \
tile \
--input_dir=../../../dataset/her2st/preprocessed_data/ST-patches/A1 \
--output_dir=../../../dataset/her2st/preprocessed_data/hover_seg/A1 \
--mem_usage=0.1 \
--draw_dot \
--save_qupath






In [12]:
import numpy as np

# 加载.npy文件
data = np.load('Alex.npy', allow_pickle=True)

# 查看基本信息
print("数据类型:", data.dtype)           # 数组元素类型
print("数组形状:", data.shape)           # 数组维度
print("总元素数:", data.size)            # 元素总数
print("数组维度:", data.ndim)            # 维度数
print("\n数组内容:")
print(data)                              # 打印数组内容



数据类型: object
数组形状: (50,)
总元素数: 50
数组维度: 1

数组内容:
['AC009133.1' 'AC092069.1' 'ACKR1' 'ACTG2' 'APOC1' 'APOD' 'AQP1'
 'C11orf96' 'C1QC' 'CCL18' 'CCL2' 'CD52' 'CILP' 'COL15A1' 'CST1' 'CXCL10'
 'CXCL11' 'CXCL9' 'FABP7' 'H19' 'HBA1' 'HBA2' 'HBB' 'HTRA3' 'IGHA1'
 'IGHG1' 'IGHG2' 'IGHG3' 'IGHM' 'IGLC1' 'IGLC2' 'IGLC3' 'LRRC75A'
 'MAP3K12' 'MMACHC' 'MMP11' 'MMP13' 'MMP9' 'MT1X' 'MUCL1' 'PLVAP' 'PTGDS'
 'SELE' 'SFRP4' 'SPARCL1' 'SPP1' 'SYNC' 'TPSB2' 'TRBC2' 'VWF']


图像特征提取

In [ ]:
from PIL import Image
import torch
from transformers import AutoImageProcessor, AutoModel
import os
import numpy as np

# 加载预训练模型
processor = AutoImageProcessor.from_pretrained("/d/zhoujl/my_model/model_train/phikon-v2")
model = AutoModel.from_pretrained("/d/zhoujl/my_model/model_train/phikon-v2")
model.eval()


source_patches_dir = "/d/zhoujl/my_model/dataset/Alex_NatGen/patches_nmzd"
feature_save_dir = "/d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_data/precomputed_features"
os.makedirs(feature_save_dir, exist_ok=True)


for slice_folder in os.listdir(source_patches_dir):
    slice_path = os.path.join(source_patches_dir, slice_folder)
    
    if not os.path.isdir(slice_path):
        continue
    
 
    slice_feature_dir = os.path.join(feature_save_dir, slice_folder)
    os.makedirs(slice_feature_dir, exist_ok=True)
    
   
    for image_file in os.listdir(slice_path):
        if not image_file.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff')):
            continue
        
        image_path = os.path.join(slice_path, image_file)
        
        try:
        
            image = Image.open(image_path)  
            
          
            inputs = processor(image, return_tensors="pt")
            
           
            with torch.inference_mode():
                outputs = model(**inputs)
                features = outputs.last_hidden_state[:, 0, :].squeeze().cpu().numpy()  # (1024,) shape
            
            
            feature_file = os.path.join(slice_feature_dir, f"{os.path.splitext(image_file)[0]}.npy")
            np.save(feature_file, features)
            
            print(f"Feature extracted and saved from {image_path} to {feature_file}")
        
        except Exception as e:
            print(f"Failed to process {image_path}: {e}")

print("All features extracted and saved according to the original classification!")

/d/zhoujl/anaconda3/envs/THItoGene/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-03-11 17:51:25.039536: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-03-11 17:51:25.161309: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-03-11 17:51:25.212379: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-03-11 17:51:

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2026-03-11 17:51:26.205035: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Feature extracted and saved from /d/zhoujl/my_model/dataset/Alex_NatGen/patches_nmzd/CID4535/ACCTCGAACTTATGCT-1.jpg to /d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_data/precomputed_features/CID4535/ACCTCGAACTTATGCT-1.npy
Feature extracted and saved from /d/zhoujl/my_model/dataset/Alex_NatGen/patches_nmzd/CID4535/CAGACACCGATCGCTG-1.jpg to /d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_data/precomputed_features/CID4535/CAGACACCGATCGCTG-1.npy
Feature extracted and saved from /d/zhoujl/my_model/dataset/Alex_NatGen/patches_nmzd/CID4535/TGCTATGGCAAAGGGA-1.jpg to /d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_data/precomputed_features/CID4535/TGCTATGGCAAAGGGA-1.npy
Feature extracted and saved from /d/zhoujl/my_model/dataset/Alex_NatGen/patches_nmzd/CID4535/CACTGACGATTGTGGA-1.jpg to /d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_data/precomputed_features/CID4535/CACTGACGATTGTGGA-1.npy
Feature extracted and saved from /d/zhoujl/my_model/dataset/Alex_NatGen/patches_nmzd/CID

In [ ]:
from PIL import Image
import torch
import os
import numpy as np
import torch.nn as nn
import torchvision.models as models
import torch.nn.functional as F
import torchvision.transforms as transforms
from tqdm import tqdm
import glob

class ImageEncoder(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = models.resnet50(pretrained=True)
        self.model = nn.Sequential(*list(self.model.children())[:-1])
        
        for p in self.model.parameters():
            p.requires_grad = False

    def forward(self, x):
        if x.dim() == 3:
            x = x.unsqueeze(0)
        
        x = self.model(x)
        x = F.adaptive_avg_pool2d(x, (1, 1))
        x = x.view(x.size(0), -1)
        return x

preprocess = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

model = ImageEncoder()
model.eval()

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"使用设备: {device}")

source_patches_dir = "/d/zhoujl/my_model/dataset/Alex_NatGen/patches_nmzd"
feature_save_dir = "/d/zhoujl/my_model/dataset/Alex_NatGen/preprocessed_data/res_features"

os.makedirs(feature_save_dir, exist_ok=True)

slice_folders = [f for f in os.listdir(source_patches_dir) 
                 if os.path.isdir(os.path.join(source_patches_dir, f))]

for slice_folder in tqdm(slice_folders, desc="处理切片文件夹"):
    slice_path = os.path.join(source_patches_dir, slice_folder)
    
    slice_feature_dir = os.path.join(feature_save_dir, slice_folder)
    os.makedirs(slice_feature_dir, exist_ok=True)
    
    image_files = [f for f in os.listdir(slice_path) 
                  if f.lower().endswith(('.png', '.jpg', '.jpeg', '.tif', '.tiff'))]
    
    if not image_files:
        print(f"警告: 在文件夹 {slice_path} 中未找到图像文件")
        continue
    
    for image_file in tqdm(image_files, desc=f"处理 {slice_folder} 中的图像"):
        image_path = os.path.join(slice_path, image_file)
        
        try:
            image = Image.open(image_path).convert('RGB')
            
            input_tensor = preprocess(image)
            input_batch = input_tensor.unsqueeze(0).to(device)
            
            with torch.no_grad():
                features = model(input_batch)
                features = features.squeeze(0).cpu().numpy()
            
            feature_file_name = os.path.splitext(image_file)[0] + ".npy"
            feature_file_path = os.path.join(slice_feature_dir, feature_file_name)
            
            np.save(feature_file_path, features)
            
        except Exception as e:
            print(f"处理图像 {image_path} 时出错: {e}")
            empty_features = np.zeros(2048)
            np.save(feature_file_path, empty_features)

print("所有特征提取并保存完成，保持原文件夹结构!")

/d/zhoujl/anaconda3/envs/THItoGene/lib/python3.9/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/d/zhoujl/anaconda3/envs/THItoGene/lib/python3.9/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


使用设备: cuda


处理切片文件夹: 100%|██████████| 6/6 [03:35<00:00, 35.92s/it]

所有特征提取并保存完成，保持原文件夹结构!


spot类型选择

In [ ]:
import os
import pandas as pd

directory_path = '/d/zhoujl/my_model/dataset/her2st/preprocessed_data/ST-patches'
all_items = os.listdir(directory_path)

sample_list = [item for item in all_items if os.path.isdir(os.path.join(directory_path, item))]
print(len(sample_list))
print(sample_list[0])

file_extension = '.tsv'

all_types = ['nolabe', 'necros', 'neopla', 'inflam', 'connec', 'no-neo']

def calculate_proportions(file_path):
    df = pd.read_csv(file_path, sep='\t')
    total_count = len(df)
    
    proportions = df['name'].value_counts(normalize=True) * 100
    
    for name_type in all_types:
        if name_type not in proportions:
            proportions[name_type] = 0.0
            
    return proportions.to_dict(), total_count

def save_proportions_to_tsv(proportions, total_count, output_path):
    with open(output_path, 'w') as f:
        f.write('name\tproportion\ttotal_count\n')
        for name in all_types:
            proportion = proportions.get(name, 0.0)
            f.write(f'{name}\t{proportion}\t{total_count}\n')

def summarize_spot_types(output_dir, summary_output_path):
    spot_types = []
    
    for filename in os.listdir(output_dir):
        if filename.endswith('_proportions.tsv'):
            file_path = os.path.join(output_dir, filename)
            
            df = pd.read_csv(file_path, sep='\t')
            
            max_proportion_row = df[df['proportion'] == df['proportion'].max()]
            max_proportion_name = max_proportion_row.iloc[0]['name']
            
            spot_name = os.path.splitext(filename)[0].replace('_proportions', '')
            
            spot_types.append({'spot': spot_name, 'type': max_proportion_name})
    
    summary_df = pd.DataFrame(spot_types)
    summary_df.to_csv(summary_output_path, sep='\t', index=False)

for sample in sample_list:
    input_dir = '/d/zhoujl/my_model/dataset/her2st/preprocessed_data/hover_seg/'+sample+'/qupath/'
    output_dir = '/d/zhoujl/my_model/dataset/her2st/preprocessed_data/spots_type/'+sample+'/'
    os.makedirs(output_dir, exist_ok=True)

    for filename in os.listdir(input_dir):
        if filename.endswith(file_extension):
            file_path = os.path.join(input_dir, filename)
            
            proportions, total_count = calculate_proportions(file_path)
            
            output_filename = os.path.splitext(filename)[0] + '_proportions.tsv'
            output_path = os.path.join(output_dir, output_filename)
            
            save_proportions_to_tsv(proportions, total_count, output_path)
            print(f"Processed {filename} and saved results to {output_path}")

    summary_output_path = os.path.join(output_dir, 'summary_spot_types.tsv')
    summarize_spot_types(output_dir, summary_output_path)
    print(f"Summary of spot types saved to {summary_output_path}")

36
G1
Processed 5x9.tsv and saved results to /d/zhoujl/my_model/dataset/her2st/preprocessed_data/spots_type/G1/5x9_proportions.tsv
Processed 22x26.tsv and saved results to /d/zhoujl/my_model/dataset/her2st/preprocessed_data/spots_type/G1/22x26_proportions.tsv
Processed 12x18.tsv and saved results to /d/zhoujl/my_model/dataset/her2st/preprocessed_data/spots_type/G1/12x18_proportions.tsv
Processed 16x18.tsv and saved results to /d/zhoujl/my_model/dataset/her2st/preprocessed_data/spots_type/G1/16x18_proportions.tsv
Processed 13x20.tsv and saved results to /d/zhoujl/my_model/dataset/her2st/preprocessed_data/spots_type/G1/13x20_proportions.tsv
Processed 7x25.tsv and saved results to /d/zhoujl/my_model/dataset/her2st/preprocessed_data/spots_type/G1/7x25_proportions.tsv
Processed 23x14.tsv and saved results to /d/zhoujl/my_model/dataset/her2st/preprocessed_data/spots_type/G1/23x14_proportions.tsv
Processed 14x19.tsv and saved results to /d/zhoujl/my_model/dataset/her2st/preprocessed_data/spot